# Hands-on 39 (Bab 15): Menambahkan Notifikasi Sederhana

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 15.

Sistem tidak hanya mencatat anomali, tetapi mengabarkannya. Dua kanal sederhana sesuai
buku: publish ke topik `capstone/iotnode01/alert` (sudah dibangun di Hands-on 38) dan
listener notifikasi yang menampilkan alert serta menyimpannya ke CSV - titik ekstensi
menuju Telegram/email/WA.

## Tujuan

Pada latihan ini, pembaca akan menambahkan notifikasi sederhana ke proyek akhir, sehingga sistem tidak hanya mencatat anomali, tetapi juga mengabarkan bahwa ada kondisi yang perlu perhatian.

## Strategi notifikasi yang dipakai

Untuk tahap buku, strategi notifikasi tidak perlu rumit. Kita gunakan dua cara sederhana:

- publish ke topik MQTT capstone/iotnode01/alert
- simpan alert ke measurement InfluxDB khusus

Dengan pendekatan ini, pembaca bisa:

- melihat alert di terminal,
- subscribe alert dengan mosquitto_sub,
- menampilkan alert di Grafana,
- dan nanti mengembangkannya ke Telegram/email/WA bila mau.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

# --- Utilitas notebook: menjalankan skrip buku APA ADANYA (tanpa flag) ---
# Skrip buku memakai while True / loop_forever, jadi notebook menghentikannya
# dari luar setelah beberapa detik dan menyimpan keluaran parsialnya.
import subprocess, sys, time
from pathlib import Path

LOG_DIR = Path("outputs/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def latar(script, nama_log, args=()):
    """Jalankan skrip buku di latar belakang (python -u), keluaran ke file log."""
    f = open(LOG_DIR / nama_log, "w", encoding="utf-8")
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=f, stderr=subprocess.STDOUT)
    return p, f

def henti(*pasangan):
    """Hentikan proses latar belakang dan tutup file lognya."""
    for p, f in pasangan:
        p.terminate()
        try:
            p.wait(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
        f.close()

def jalankan_buku(script, detik, args=()):
    """Jalankan skrip buku selama `detik` detik lalu hentikan; kembalikan keluaran."""
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        out, _ = p.communicate(timeout=detik)
    except subprocess.TimeoutExpired:
        p.terminate()
        try:
            out, _ = p.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
            out, _ = p.communicate()
    return out

def log_teks(nama_log):
    return (LOG_DIR / nama_log).read_text(encoding="utf-8")

# --- Utilitas InfluxDB (nilai bawaan skrip buku) ---
from influxdb_client import InfluxDBClient
INFLUX_URL, INFLUX_TOKEN = "http://localhost:8086", "GANTI_DENGAN_TOKEN_ANDA"
INFLUX_ORG, INFLUX_BUCKET = "iotlab", "sensor_data"
_ic = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
if not _ic.ping():
    raise RuntimeError("InfluxDB belum berjalan di localhost:8086 (org iotlab, bucket sensor_data, "
                       "token GANTI_DENGAN_TOKEN_ANDA). Skrip buku di notebook ini menulis ke InfluxDB.")

def influx_df(measurement, rentang="-30d"):
    import pandas as pd
    q = (f'from(bucket:"{INFLUX_BUCKET}") |> range(start: {rentang}, stop: 1d) '
         f'|> filter(fn:(r) => r._measurement == "{measurement}") '
         f'|> pivot(rowKey:["_time"], columnKey:["_field"], valueColumn:"_value")')
    res = _ic.query_api().query_data_frame(q)
    df = pd.concat(res) if isinstance(res, list) else res
    if len(df):
        df = df.drop(columns=[c for c in df.columns if c in ("result", "table", "_start", "_stop", "_measurement")])
        df = df.sort_values("_time").reset_index(drop=True)
    return df

def influx_hapus(measurement):
    _ic.delete_api().delete("1970-01-01T00:00:00Z", "2100-01-01T00:00:00Z",
                            f'_measurement="{measurement}"', bucket=INFLUX_BUCKET, org=INFLUX_ORG)


[INFO] Working directory: /downloads/iot-ml-book


## 1. Jalankan Sistem dengan Notifikasi

In [2]:
influx_hapus("capstone_device_monitoring"); influx_hapus("capstone_alerts")
broker = subprocess.Popen([sys.executable, "scripts/mqtt/run_broker.py"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
try:
    listener = latar("scripts/capstone_alert_listener.py", "capstone_listener40_nb.log", ("--max-messages", "2", "--timeout", "120"))
    sub = latar("scripts/subscribers/capstone_subscriber_with_alert.py", "capstone_sub40_nb.log")
    time.sleep(2)
    pub = latar("scripts/publishers/capstone_publisher.py", "capstone_pub40_nb.log")
    for _ in range(60):          # tunggu sampai listener menerima 2 notifikasi (maks. 2 menit)
        time.sleep(2)
        if listener[0].poll() is not None:
            break
    henti(pub, sub, listener)
    print("--- Output listener notifikasi ---")
    print(log_teks("capstone_listener40_nb.log"))
finally:
    broker.terminate()

--- Output listener notifikasi ---
[INFO] Menunggu alert di topik: capstone/iotnode01/alert

=== NOTIFIKASI ALERT ===
Waktu   : 2026-11-02T12:20:48
Device  : iotnode01
Alasan  : iforest
Daya    : 176.86 W
Arus    : 3.55 A

=== NOTIFIKASI ALERT ===
Waktu   : 2026-11-02T12:20:58
Device  : iotnode01
Alasan  : threshold,iforest
Daya    : 340.7687297483094 W
Arus    : 13.645603611378863 A

[SUKSES] 2 notifikasi disimpan ke outputs/reports/capstone_notifications.csv



## 2. Verifikasi Log Notifikasi

In [3]:
import pandas as pd
notif = pd.read_csv("outputs/reports/capstone_notifications.csv")
print(notif.to_string(index=False))

          timestamp device_id  temperature_c  humidity_pct  power_watt  current_amp            reason
2026-11-02T12:20:48 iotnode01          33.62         40.27   176.86000     3.550000           iforest
2026-11-02T12:20:58 iotnode01          25.14         50.21   340.76873    13.645604 threshold,iforest


## Panduan Lengkap dari Buku

_Bagian ini dikutip dari buku PDF (sumber kebenaran praktikum) dan diformat ulang agar mudah dibaca. Kode asli buku tidak diubah; bagian instalasi/persiapan lingkungan diringkas karena sudah dijelaskan di README._

**3. Modifikasi subscriber agar mengirim alert**

Buat file scripts/subscribers/capstone_subscriber_with_alert.py:

```python
 import json
 from pathlib import Path
 import joblib
 import pandas as pd
 import paho.mqtt.client as mqtt
 from influxdb_client import InfluxDBClient, Point, WritePrecision
 from influxdb_client.client.write_api import SYNCHRONOUS
 BROKER = "localhost"
 PORT = 1883
 TOPIC_INPUT = "capstone/iotnode01/telemetry"
 TOPIC_ALERT = "capstone/iotnode01/alert"
 MODEL_PATH = Path("data/models/capstone_iforest.joblib")
 INFLUX_URL = "http://localhost:8086"
 INFLUX_TOKEN = "GANTI_DENGAN_TOKEN_ANDA"
 INFLUX_ORG = "iotlab"
 INFLUX_BUCKET = "sensor_data"
 feature_cols = ["temperature_c", "humidity_pct", "power_watt",
 "current_amp"]
 model = joblib.load(MODEL_PATH)
influx_client = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN,
org=INFLUX_ORG)
write_api = influx_client.write_api(write_options=SYNCHRONOUS)
def validate_payload(payload):
    required = ["timestamp", "device_id", "temperature_c",
"humidity_pct", "power_watt", "current_amp"]
    for field in required:
        if field not in payload:
            raise ValueError(f"Field hilang: {field}")
def build_features(payload):
    return pd.DataFrame([{
        "temperature_c": float(payload["temperature_c"]),
        "humidity_pct": float(payload["humidity_pct"]),
        "power_watt": float(payload["power_watt"]),
        "current_amp": float(payload["current_amp"]),
    }])
def on_connect(client, userdata, flags, reason_code, properties=None):
    client.subscribe(TOPIC_INPUT, qos=1)
    print("Subscribed to", TOPIC_INPUT)
def on_message(client, userdata, msg):
    try:
        payload = json.loads(msg.payload.decode())
        validate_payload(payload)
        X = build_features(payload)
         threshold_alert = int(
             X["power_watt"].iloc[0] > 280 or
             X["current_amp"].iloc[0] > 10.0
         )
         pred = model.predict(X[feature_cols])[0]
         anomaly_flag = 1 if pred == -1 else 0
         is_alert = threshold_alert == 1 or anomaly_flag == 1
         reason = []
         if threshold_alert:
             reason.append("threshold")
         if anomaly_flag:
             reason.append("iforest")
         reason_text = ",".join(reason) if reason else "none"
        # simpan data monitoring biasa
        point = (
            Point("capstone_device_monitoring")
            .tag("device_id", payload["device_id"])
            .field("temperature_c", float(payload["temperature_c"]))
            .field("humidity_pct", float(payload["humidity_pct"]))
            .field("power_watt", float(payload["power_watt"]))
            .field("current_amp", float(payload["current_amp"]))
            .field("iforest_anomaly", anomaly_flag)
            .field("threshold_alert", threshold_alert)
            .time(payload["timestamp"], WritePrecision.S)
        )
        write_api.write(bucket=INFLUX_BUCKET, org=INFLUX_ORG,
record=point)
         if is_alert:
             alert_payload = {
                 "timestamp": payload["timestamp"],
                 "device_id": payload["device_id"],
                 "temperature_c": payload["temperature_c"],
                 "humidity_pct": payload["humidity_pct"],
                 "power_watt": payload["power_watt"],
                 "current_amp": payload["current_amp"],
                 "reason": reason_text
             }
               client.publish(TOPIC_ALERT, json.dumps(alert_payload),
qos=1)
               print("ALERT:", alert_payload)
            alert_point = (
                Point("capstone_alerts")
                .tag("device_id", payload["device_id"])
                .tag("reason", reason_text)
                .field("temperature_c",
float(payload["temperature_c"]))
                .field("humidity_pct", float(payload["humidity_pct"]))
                .field("power_watt", float(payload["power_watt"]))
                .field("current_amp", float(payload["current_amp"]))
                .field("alert_flag", 1)
                .time(payload["timestamp"], WritePrecision.S)
            )
            write_api.write(bucket=INFLUX_BUCKET, org=INFLUX_ORG,
record=alert_point)
     except Exception as exc:
         print("Error:", exc)
client = mqtt.Client(client_id="capstone-subscriber-alert")
client.on_connect = on_connect
client.on_message = on_message
client.connect(BROKER, PORT, 60)
client.loop_forever()
```

Jalankan:

```python
python scripts/subscribers/capstone_subscriber_with_alert.py
```

**4. Verifikasi alert di MQTT**

Buka terminal baru:

```python
mosquitto_sub -h localhost -t "capstone/iotnode01/alert" -v
```

Jika kondisi abnormal terjadi, Anda akan melihat payload alert keluar.

**5. Tampilkan alert di Grafana**

Tambahkan panel dari measurement:

- capstone_alerts

Panel yang bagus

- jumlah alert per jam
- tabel alert terbaru
- panel status merah bila ada alert baru
- filter berdasarkan reason

Tujuannya

Agar operator tidak hanya melihat data mentah, tetapi langsung melihat event penting yang perlu tindakan.

**6. Insight Hands-on 39**

Latihan ini mengajarkan bahwa:

- notifikasi sederhana sangat cukup untuk tahap awal,
- kombinasi threshold + model sering lebih berguna daripada salah satu saja,
- dan alert sebaiknya ditujukan untuk aksi operasional, bukan sekadar log

tambahan.

## Penutup

Notifikasi sampai ke listener melalui topik alert dan tersimpan di
`outputs/reports/capstone_notifications.csv`. Listener ini adalah titik ekstensi yang
jelas: ganti aksi cetak/simpan dengan pengiriman ke Telegram, email, atau WhatsApp.
Di Grafana, alert bisa ditampilkan dari measurement `capstone_alerts` saat memakai
storage InfluxDB (`--influx`).